# Jam Transformer — Colab Smoke Test
End-to-end pipeline check on the free Colab T4 tier, using synthetic data so no MIDI dataset upload is needed.

Steps:
1. Upload / clone the `project_transformer` folder.
2. Install minimal deps.
3. Generate 32 synthetic songs → tokenize.
4. Train for 2 epochs (`<60 s` on T4).
5. Sample an accompaniment for one of the synthetic melodies.
6. Print the resulting MIDI summary.

## 1. Set up the project folder
If you uploaded the folder via the side panel, set `PROJECT_DIR` accordingly. If it lives on Google Drive, mount the drive first.

In [ ]:
import os, sys, pathlib

# Adjust this if the project lives elsewhere (e.g. on Drive).
PROJECT_DIR = '/content/project_transformer'

assert pathlib.Path(PROJECT_DIR).exists(), f'Upload project_transformer to {PROJECT_DIR} first.'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(pathlib.Path(PROJECT_DIR) / 'src'))
print('CWD:', os.getcwd())

In [ ]:
!pip install -q miditoolkit loguru pytorch-lightning>=2.2 pretty_midi

## 2. Tokenize synthetic songs

In [ ]:
!python scripts/prepare_data.py --synthetic --num_songs 32 --out_dir data/processed

## 3. Smoke training (2 epochs, tiny batch)
If this runs end-to-end without crashing, the pipeline is ready for a paid GPU.

In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'
!python scripts/train.py --epochs 2 --batch_size 4

## 4. Sample an accompaniment for one synthetic melody
We first dump one synthetic melody as a MIDI file (using `events_to_midi`), then run inference on it.

In [ ]:
from pathlib import Path
from jam_transformer.config import load_config
from jam_transformer.midi_io import events_to_midi
import sys
sys.path.insert(0, 'scripts')
from prepare_data import _synthesize_song

cfg = load_config('configs/config.yaml')
events, tempo = _synthesize_song(seed=999)
melody_only = [e for e in events if e.track == 'melody']
midi = events_to_midi(melody_only, cfg.tokenizer, tempo_bpm=tempo)
Path('output').mkdir(exist_ok=True)
midi.dump('output/prompt_melody.mid')
print('Wrote output/prompt_melody.mid')

In [ ]:
!ls checkpoints/
!python scripts/inference.py \
    --checkpoint checkpoints/last.ckpt \
    --melody_midi output/prompt_melody.mid \
    --output output/accompaniment.mid \
    --temperature 1.0 --top_k 64 --top_p 0.95 --max_new_tokens 256

In [ ]:
import miditoolkit
m = miditoolkit.MidiFile('output/accompaniment.mid')
for inst in m.instruments:
    print(f'{inst.name}: {len(inst.notes)} notes')
    for n in inst.notes[:5]:
        print(f'  pitch={n.pitch} start={n.start} end={n.end} vel={n.velocity}')

## 5. (Optional) Render to audio
Upload a `.sf2` soundfont (e.g. *GeneralUser GS*) and set `inference.render_audio: true` + `inference.soundfont: "<path>"` in `configs/config.yaml`.